# Normalized & Imputate data

## Import Necessary library

In [2]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
import pandas as pd
import numpy as np
import lightgbm as lbm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, mean_squared_log_error

from utils.preprocess.normalized.cpu import resolve_cpu_features
from utils.preprocess.normalized.mem_require import resolve_mem_require
from utils.preprocess.normalized.os import resolve_os_features
from utils.preprocess.normalized.price import resolve_price
from utils.preprocess.imputation.genres_tags.func import retrieve_tag_list
from utils.preprocess.normalized.release_date import resolve_release_date
from utils.preprocess.imputation.genres_tags.schemas import RetrieveRequest

### Load Data

In [4]:
DATA_PATH = "data/output/SteamGames_Clean_1.csv"
df = pd.read_csv(DATA_PATH)
df.head(5)

,Appid,Name,Type,ReleaseDate,Genres,Developers,Publishers,Description,price,Thumbnail,Tags,ReviewScore,PositiveReview,NegativeReview,OsRequirement,MemoryRequirement,CpuRequirement,Rank
0,3764200,Resident Evil Requiem,game,"Feb 26, 2026","Action,Adventure","CAPCOM Co., Ltd.","CAPCOM Co., Ltd.",Resident Evil Requiem Deluxe Edition Resident ...,69.99$,https://shared.akamai.steamstatic.com/store_it...,NaN,9,23715,793,Windows 11 (64bit required),16 GB RAM,Intel corei5-8500 / AMD Ryzen 5 3500,1
1,730,Counter-Strike 2,game,"Aug 21, 2012","Action,Free To Play",Valve,Valve,"For over two decades, Counter-Strike has offer...",0$,https://shared.akamai.steamstatic.com/store_it...,"FPS,Shooter,Multiplayer,Competitive,Action,Tea...",8,1214897,202393,Windows® 10,8 GB RAM,4 hardware CPU threads - Intel® Core™ i5 750 o...,2
2,1808500,ARC Raiders,game,"Oct 30, 2025",Action,Embark Studios,Embark Studios,ARC Raiders Deluxe Edition ARC Raiders Deluxe ...,39.99$,https://shared.akamai.steamstatic.com/store_it...,"Extraction Shooter,Multiplayer,PvP,PvE,Third-P...",8,149531,23194,Windows 10 or later 64-bit (latest update),12 GB RAM,Intel Core i5-6600K or AMD Ryzen R5 1600 proce...,3
3,359550,Tom Clancy's Rainbow Six Siege,game,"Dec 1, 2015","Action,Free To Play",Ubisoft Montreal,Ubisoft,Season 11 Year 1 Overview Year 11 Roadmap Free...,0$,https://shared.akamai.steamstatic.com/store_it...,NaN,8,522886,106839,"Windows 10, Windows 11 (64-bit versions)",8 GB RAM,AMD Ryzen 3 3100 / Intel i3-8100,4
4,1172470,Apex Legends™,game,"Nov 4, 2020","Action,Adventure,Free To Play",Respawn,Electronic Arts,Apex Legends: Breach About the Game Conquer wi...,0$,https://shared.akamai.steamstatic.com/store_it...,"Free to Play,Multiplayer,Battle Royale,FPS,Sho...",6,820,216,64-bit Windows 10,6 GB RAM,"AMD FX 4350 or Equivalent, Intel Core i3 6300 ...",5


## Normalized data
### MemoryRequirment
`MemoryRequirement` refers to minimum RAM to play the game. We need to remove the word **RAM** at the end of each word and normalized all unit back to GB


In [5]:
df['MemoryRequirement'] = df['MemoryRequirement'].apply(
    lambda mem : resolve_mem_require(mem) if pd.notna(mem) else pd.NA
).astype('Float64')

### OsRequirement

In [6]:
os_nan = pd.Series([pd.NA, pd.NA], index=['OS_ID', 'OS_Tier'])

df[['OS_ID', 'OS_Tier']] = df['OsRequirement'].apply(
    lambda os : resolve_os_features(os) if pd.notna(os) else os_nan
)

### ReleaseDate

In [7]:
df['ReleaseDate'] = df['ReleaseDate'].apply(
    lambda date : resolve_release_date(date)
).astype('Int64')

### Price

In [8]:
df['price'] = df['price'].apply(
    lambda price : resolve_price(price)
).astype('Float64')

## Impuatation

### Impuate CpuRequirement

In [9]:
core_nan = pd.Series([pd.NA, pd.NA], index=['CPU_Rank', 'CPU_Cores'])

df[['CPU_Rank', 'CPU_Cores']] = df['CpuRequirement'].apply(
    lambda cpu: resolve_cpu_features(cpu) if pd.notna(cpu) else core_nan
)

### Impuate Tags & Genres

In [10]:
def impute_tags_for_row(row):
    # Keep existing tags if already present
    if pd.notna(row['Tags']):
        return row['Tags']

    # Handle potentially missing Genres or Description
    genres_list = row['Genres'].split(',') if pd.notna(row['Genres']) else []
    desc_text = str(row['Description']) if pd.notna(row['Description']) else ""

    response = retrieve_tag_list(RetrieveRequest(
        name=str(row['Name']),
        genres=genres_list,
        description=desc_text,
        top_k=10,
        top_tag=10,
        top_genre=3
    ))

    # Convert list of tags back into a comma-separated string for the CSV
    return ",".join(response.tag_list) if response.tag_list else None

def impute_genres_for_row(row):
    if pd.notna(row['Genres']):
        return row['Genres']

    game_name = row['Name'] if pd.notna(row['Name']) else ""
    desc_text = str(row['Description']) if pd.notna(row['Description']) else ""
    tags_list = row['Tags'].split(',') if pd.notna(row['Tags']) else []

    response = retrieve_tag_list(RetrieveRequest(
        name=game_name,
        description=desc_text,
        genres=tags_list,           # treat tags game as genres
        top_k=10,
        top_tag=10,
        top_genre=4,
    ))

    return ','.join(response.genres_list) if response.tag_list else None

# Apply across rows (axis=1 is required)
df['Tags'] = df.apply(impute_tags_for_row, axis=1)
df['Genres'] = df.apply(impute_genres_for_row, axis=1)

We will not impute any further because **performance scores on those tests was poor** and could not be used to predict missing values. We left about **3506 rows contains missing values**( which is \~11% of the dataset)

In [37]:
df.isna().any(axis=1).sum()

np.int64(3506)

In [38]:
df.to_csv('data/output/SteamGames_Clean_2.csv', index=False)